# Apache Flume

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/flume.ipynb)

Apache Flume is a distributed, reliable, and available service for efficiently collecting, aggregating, and moving large amounts of log and event data into HDFS, HBase, Kafka, Solr, or another agent. An agent is a JVM process that wires a source to one or more channels and a sink, and the channel is what makes every handoff transactional.

```text
source -> interceptors -> channel selector -> channel (buffer) -> sink -> storage / next agent
```

Key ideas:

+ A source writes events into a channel; it never writes directly to HDFS.
+ A channel is the durability boundary: memory for speed, file or JDBC for survival across restarts.
+ A sink removes an event from the channel only after the destination acknowledges the batch.
+ The unit of transfer is a transaction; reliability comes from commit/rollback across source, channel, and sink.

## Core model

### Events

An event is `{ headers: Map<String,String>, body: byte[] }`. Headers are small strings copied with the event; the body is the payload and Flume never interprets it. Sources that read text still hand the sink bytes, which is why a serializer or a record processor is needed to change format.

Body size and event count both matter. A `netcat` source creates one event per line, a spooling directory source creates one event per file, and a million 1 KB events behaves very differently from one 1 GB event. A memory channel counts events in `capacity`, so `byteCapacity` is what actually protects the heap.

Common mistake: assuming headers are free. Every header is written into the file channel record and, for sequence files, into the HDFS metadata. Hundreds of headers per event inflate storage and slow the write path, so extract only what routing and partitioning need.

```text
Event {
  headers: { host: web-07, timestamp: 1759600000000 },
  body:    "2026-10-04T13:20:34Z INFO checkout order=88217 amount=42.10"
}
```

### Agents and components

Source, channel, and sink, plus the optional interceptors, channel selectors, and sink processors. A source has one or more channels, a sink has exactly one channel or belongs to a sink group, and the agent is the process that hosts all of them.

The path in order: the source reads and puts, interceptors transform in the source thread, the selector chooses channels, the channel stores, the sink-group processor chooses a sink, the sink takes and writes, then commits. Knowing where each step runs is what makes a bottleneck visible.

Every config has the same skeleton, and every skeleton has the same three links: the component lists, `r1.channels = c1` on the source, and `k1.channel = c1` on the sink. Missing any one of the three is the most common startup error in a new agent.

```properties
a1.sources = r1
a1.channels = c1
a1.sinks = k1

a1.sources.r1.type = netcat
a1.sources.r1.bind = 0.0.0.0
a1.sources.r1.port = 44444
a1.sources.r1.channels = c1

a1.channels.c1.type = memory
a1.channels.c1.capacity = 100000

a1.sinks.k1.type = logger
a1.sinks.k1.channel = c1
```

Common mistake: running source, channel, and sink in one JVM and assuming the JVM is not a failure domain. An out-of-memory error in the source takes the sink with it. On a busy host, separate the collection agent from the delivery agent so a slow destination cannot stall log collection.

```bash
# Start one agent named a1 with a properties config file
flume-ng agent --conf /etc/flume/conf --conf-file /etc/flume/conf/weblogs.conf -n a1 \
  -Dflume.root.logger=INFO,console
```

### Transactions and at-least-once delivery

Mechanically an agent is one JVM with three groups of threads. Source threads pull from the outside world and call `channel.put(event)` inside a transaction. Sink threads call `channel.take()` for a batch and commit only after the destination acknowledges the write. A rollback returns the batch to the channel, so the next attempt re-delivers the same events instead of losing them.

Worked example. A source with `batchSize = 500` and a channel with `transactionCapacity = 500` commits one batch per 500 events; at 20,000 events/s that is 40 transactions per second. Raise `transactionCapacity` to 5,000 and the same load is 4 large transactions per second, which is far cheaper in fsyncs for a file channel.

```properties
a1.sources.r1.batchSize = 500
a1.channels.c1.capacity = 200000
a1.channels.c1.transactionCapacity = 5000
```

Common mistake: reading "reliable" as "exactly once". Flume is at-least-once. When the destination write succeeds but the acknowledgement is lost, the sink re-sends the batch. Make the destination idempotent, for example a deterministic file name or an upsert on a business key, instead of expecting Flume to deduplicate.

Between two agents there are two transactions, one on each side. The sending sink takes a batch, writes it to the receiving source, and commits only when the receiver's channel transaction commits and returns success. An acknowledgement lost on the way back causes a re-send, which is exactly the at-least-once boundary.

```text
Producer agent                         Consumer agent
  source -> channel                      channel -> sink
     |  begin txn                            |  begin txn
     |  write events                         |  write events
     |  commit  ----------------------------->|  commit
     |  (waits for ack)          ack          |  (returns success)
```

Worked failure trace. The edge sink takes 2,000 events, the collector commits them, the acknowledgement is lost, and the edge sink times out and rolls back. It re-sends the same 2,000 events, so the collector now holds 4,000 where the source produced 2,000. Duplicate handling is therefore a downstream responsibility.

Common mistake: assuming the two transactions commit atomically together. They do not; there is a window where the receiver has committed and the sender has not. Any design that needs exactly once must add an idempotency key or a deduplication step on the destination.

The cells below model a channel and a sink in plain Python: one lost acknowledgement turns into duplicates, never into loss.

In [ ]:
from collections import deque


class ChannelFullException(Exception):
    pass


class Channel:
    """A Flume-style channel: bounded queue with put/take transactions."""

    def __init__(self, capacity, transaction_capacity, durable):
        self.capacity = capacity
        self.transaction_capacity = transaction_capacity
        self.durable = durable  # file channel = True, memory channel = False
        self.events = deque()
        self.in_flight = []     # taken by a sink but not yet committed

    def put(self, batch):
        if len(batch) > self.transaction_capacity:
            raise ValueError("batch larger than transactionCapacity")
        if len(self.events) + len(self.in_flight) + len(batch) > self.capacity:
            raise ChannelFullException("Unable to put batch on required channel")  # back-pressure
        self.events.extend(batch)

    def take(self, n):
        n = min(n, self.transaction_capacity, len(self.events))
        self.in_flight = [self.events.popleft() for _ in range(n)]
        return list(self.in_flight)

    def commit(self):
        self.in_flight = []

    def rollback(self):
        self.events.extendleft(reversed(self.in_flight))  # the batch is re-delivered next time
        self.in_flight = []

    def crash(self):
        """JVM dies: a file channel replays from its log, a memory channel loses everything."""
        if self.durable:
            self.rollback()
        else:
            self.events.clear()
            self.in_flight = []


def run_sink(channel, destination, batch_size, lose_ack_on=()):
    """Write each batch, then commit. A lost acknowledgement means rollback and re-send."""
    attempt = 0
    while channel.events:
        batch = channel.take(batch_size)
        destination.extend(batch)  # the write itself succeeds...
        attempt += 1
        if attempt in lose_ack_on:
            channel.rollback()     # ...but the sink never hears back, so it retries the batch
        else:
            channel.commit()

In [ ]:
events = [{"id": i, "body": f"line {i}"} for i in range(1_000)]

channel = Channel(capacity=5_000, transaction_capacity=100, durable=True)
for start in range(0, len(events), 100):
    channel.put(events[start:start + 100])
destination = []
run_sink(channel, destination, batch_size=100, lose_ack_on={2})

distinct = {event["id"] for event in destination}
assert len(destination) == 1_100 and len(distinct) == 1_000  # at-least-once: nothing lost, 100 duplicated
idempotent = {event["id"]: event for event in destination}   # upsert on a business key removes them
assert len(idempotent) == len(events)
print(f"delivered {len(destination)} rows, {len(distinct)} distinct; idempotent sink keeps {len(idempotent)}")

## Channels

### Channel types

| Channel | Speed | Durability | Typical use |
|---|---|---|---|
| memory | highest | none across restart | high-volume, loss-tolerant logs |
| file | medium | high | audit, billing, security logs |
| jdbc | low | high | small legacy pipelines |
| kafka | medium-high | high | decoupled multi-consumer ingestion |
| spillable memory | high until spill | medium | bursty traffic with a disk safety net |

A memory channel is a `LinkedBlockingDeque` on the heap. A file channel is an append-only data log plus a checkpoint. A JDBC channel is a Derby-backed transactional queue. A Kafka channel is a topic that acts as the queue, so the sink commits an offset rather than a local transaction. A spillable memory channel keeps events on heap first and spills to a file channel once `capacity` is reached, up to `overflowCapacity`.

Worked numbers for one stream of 1 KB events at 50,000 events/s. A memory channel handles it in one agent with a few GB of heap. A file channel on a single spinning disk fsync-throttles far below that unless `fsyncPerTransaction = false` or the disk is NVMe. A Kafka channel moves the durability to the brokers and scales by partition. Match the channel to the failure you must survive.

Common mistake: choosing the spillable memory channel as a free upgrade. Events still on heap die with the JVM, only the spilled portion is durable, and only that part is restored after a restart. It is a burst absorber, not a durable channel; if loss is unacceptable, use file or Kafka.

### Memory and file channel internals

The memory channel is a `LinkedBlockingDeque<Event>` on the JVM heap. `capacity` is the number of events it holds, and `byteCapacity` caps total bytes so a few large events cannot exhaust the heap. `transactionCapacity` is the largest batch a source or sink may commit in one transaction.

The file channel appends events to data files under `dataDirs` and records how far each sink has read in `checkpointDir`. It fsyncs on commit by default (`fsyncPerTransaction = true`), which is why it is roughly an order of magnitude slower than memory but survives `kill -9`. `checkpointInterval` bounds how much replay is needed after an unclean shutdown.

```properties
a1.channels.mem.type = memory
a1.channels.mem.capacity = 1000000
a1.channels.mem.transactionCapacity = 20000
a1.channels.mem.byteCapacityBufferPercentage = 20

a1.channels.file.type = file
a1.channels.file.checkpointDir = /data/flume/checkpoint
a1.channels.file.dataDirs = /data/flume/data
a1.channels.file.fsyncPerTransaction = true
a1.channels.file.checkpointInterval = 30000
```

Common mistake: setting `transactionCapacity` equal to `capacity`. While one transaction is in flight the channel has no headroom, and sources and sinks log `ChannelException: Unable to put batch on required channel`. Keep `transactionCapacity` between 1% and 5% of `capacity`.

### Which channel prevents data loss?

The file channel is the durable default: events are written to disk and the checkpoint records commit progress, so a restart replays whatever the sink never acknowledged. The JDBC channel is also durable but serializes through Derby and is far slower. The Kafka channel is durable in a topic and is the best of the three when Kafka already exists and you want the buffer to outlive the agent.

Common mistake: claiming a durable channel guarantees no loss regardless of the source. An `exec` source or a `netcat` source has no replay, so events that arrive while the channel is full are lost at the source. Durability starts at the source, not at the channel.

In [ ]:
def backlog_after_crash(durable):
    channel = Channel(capacity=1_000, transaction_capacity=100, durable=durable)
    for start in range(0, 500, 100):
        channel.put(events[start:start + 100])
    channel.take(100)  # a sink batch is in flight when the agent dies
    channel.crash()
    return len(channel.events)


assert backlog_after_crash(durable=True) == 500   # file channel: replayed, including the in-flight batch
assert backlog_after_crash(durable=False) == 0    # memory channel: the buffered events are gone

small = Channel(capacity=300, transaction_capacity=100, durable=True)
accepted = 0
try:
    for start in range(0, 1_000, 100):
        small.put(events[start:start + 100])
        accepted += 100
except ChannelFullException as exc:
    print("source blocked after", accepted, "events:", exc)
assert accepted == 300  # a replayable source waits; an exec or netcat source would drop here

## Routing

### Interceptors

Interceptors are invoked in order by the source before the event reaches a channel. An interceptor builder creates each one, `intercept(event)` returns the event possibly modified, and the batch method can return a shorter list, which is how `regex_filter` drops events.

Order matters. A typical chain is `timestamp`, then `host`, then `static`, then `regex_extractor`, then `regex_filter`. If the filter runs before the extractor, the extractor never sees the dropped events. If a multiplexing selector depends on a header, that header must be set by an interceptor that runs before the selector. Worked config with a static fallback header, so the selector always has a value:

```properties
a1.sources.r1.interceptors = i1 i2
a1.sources.r1.interceptors.i1.type = regex_extractor
a1.sources.r1.interceptors.i1.regex = ^(\\S+)\\s+(.*)$
a1.sources.r1.interceptors.i1.serializers = s1
a1.sources.r1.interceptors.i1.serializers.s1.name = level
a1.sources.r1.interceptors.i2.type = static
a1.sources.r1.interceptors.i2.key = level
a1.sources.r1.interceptors.i2.value = INFO
a1.sources.r1.interceptors.i2.preserveExisting = true
```

Common mistake: putting an expensive regex extractor in the interceptor chain of a high-volume source. Interceptors run once per event in the source thread, so one slow regex caps source throughput. Move heavy parsing to the sink path, to a record-oriented processor downstream, or remove it.

Common mistake: getting the `excludeEvents` flag backwards on `regex_filter`. The flag inverts which events survive, so a wrong value silently discards the events you meant to keep. Point the chain at a `logger` sink first and confirm the counts before writing to HDFS.

Worked chain that stamps time, tags the host, extracts a level, and drops debug lines:

```properties
a1.sources.r1.interceptors = ts host extract drop_debug
a1.sources.r1.interceptors.ts.type = timestamp
a1.sources.r1.interceptors.host.type = host
a1.sources.r1.interceptors.extract.type = regex_extractor
a1.sources.r1.interceptors.extract.regex = ^(\\w+)\\s+(.*)$
a1.sources.r1.interceptors.extract.serializers = s1
a1.sources.r1.interceptors.extract.serializers.s1.name = level
a1.sources.r1.interceptors.drop_debug.type = regex_filter
a1.sources.r1.interceptors.drop_debug.regex = ^.*DEBUG.*$
a1.sources.r1.interceptors.drop_debug.excludeEvents = true
```

### Channel selectors

The replicating selector writes every event to every channel in the source's `channels` list. Every channel must accept the event or the whole transaction fails, so replication multiplies both storage and sink capacity. It is the default whenever more than one channel is listed and no selector type is set.

The multiplexing selector maps a header value to a channel list. `selector.header` names the header, `selector.mappings.<value>` names the channels, and `selector.default` catches everything else. Events whose header matches no mapping and has no default are dropped, so always set a default.

```properties
a1.sources.r1.channels = c_err c_archive
a1.sources.r1.selector.type = multiplexing
a1.sources.r1.selector.header = level
a1.sources.r1.selector.mappings.ERROR = c_err
a1.sources.r1.selector.mappings.INFO = c_archive
a1.sources.r1.selector.default = c_archive
```

Common mistake: a multiplexing selector with no `default` and a header that is sometimes absent, for example when the source does not always emit `level`. Those events vanish with only a warning. Set a default channel, or add a `static` interceptor that supplies a fallback header before the selector runs.

In [ ]:
import re
import time

LEVEL = re.compile(r"^(ERROR|WARN|INFO|DEBUG)\s")


def timestamp_interceptor(event):
    event["headers"].setdefault("timestamp", str(int(time.time() * 1000)))
    return event


def level_extractor(event):  # regex_extractor: body -> header
    match = LEVEL.match(event["body"])
    if match:
        event["headers"]["level"] = match.group(1)
    return event


def static_level_default(event):  # static interceptor with preserveExisting = true
    event["headers"].setdefault("level", "INFO")
    return event


def multiplex(event, mappings, default=None):
    """Multiplexing selector: header value -> channel names; unmatched events go to the default."""
    return mappings.get(event["headers"].get("level"), default or [])


def run_source(lines, interceptors, mappings, default=None):
    channels = {"c_err": [], "c_archive": []}
    dropped = 0
    for line in lines:
        event = {"headers": {}, "body": line}
        for interceptor in interceptors:  # the chain runs in order, in the source thread
            event = interceptor(event)
        targets = multiplex(event, mappings, default)
        dropped += not targets
        for name in targets:
            channels[name].append(event)
    return channels, dropped


lines = ["ERROR payment declined", "INFO checkout ok", "user clicked banner", "WARN slow response"]
mappings = {"ERROR": ["c_err"], "INFO": ["c_archive"]}

_, dropped_no_default = run_source(lines, [timestamp_interceptor, level_extractor], mappings)
channels, dropped = run_source(lines, [timestamp_interceptor, level_extractor, static_level_default],
                               mappings, default=["c_archive"])

assert dropped_no_default == 2  # no header match and no default: events vanish silently
assert dropped == 0 and len(channels["c_err"]) == 1 and len(channels["c_archive"]) == 3
print("without default:", dropped_no_default, "dropped | with default:", {k: len(v) for k, v in channels.items()})

### Sink groups: failover and load balancing

There are two fan-out points. At the source, replication or multiplexing writes across channels. At the sink layer, a sink group with a load-balancing or failover processor picks one sink per batch. Source fan-out copies the event to several channels; sink-group fan-out sends each batch to exactly one sink.

Worked failover config with explicit priorities:

```properties
a1.sinkgroups = g1
a1.sinkgroups.g1.sinks = k1 k2
a1.sinkgroups.g1.processor.type = failover
a1.sinkgroups.g1.processor.priority.k1 = 10
a1.sinkgroups.g1.processor.priority.k2 = 5
a1.sinkgroups.g1.processor.maxpenalty = 30000
a1.sinkgroups.g1.processor.backoff = true
```

Common mistake: using load balancing when ordering matters. A load-balancing sink group spreads consecutive batches across sinks, so two events from the same file can land out of order. Use failover for ordered pipelines and load balancing only when the destination is order-insensitive.

### Multi-hop flows

Multi-hop uses an Avro RPC bridge. The edge agent's Avro sink takes a batch from its own channel and appends it to the collector's Avro source, which puts it into the collector's channel. The edge sink commits only when the collector's transaction commits, so a collector failure rolls the edge batch back.

```properties
# Agent on the edge: tail local logs, forward over Avro
edge.sources.r1.type = taildir
edge.sources.r1.filegroups = f1
edge.sources.r1.filegroups.f1 = /var/log/app/.*\.log
edge.sources.r1.channels = c1
edge.channels.c1.type = file
edge.channels.c1.checkpointDir = /data/flume/edge/checkpoint
edge.channels.c1.dataDirs = /data/flume/edge/data
edge.sinks.k1.type = avro
edge.sinks.k1.hostname = collector.example.com
edge.sinks.k1.port = 4141
edge.sinks.k1.channel = c1
```

Both ends must agree on host, port, and, for secured clusters, Kerberos and TLS settings. The Avro source listens with `bind` and `port`; the Avro sink connects with `hostname` and `port`, and exposes `batchSize`, `connect-timeout`, and `request-timeout` knobs for tuning the RPC.

Common mistake: many edge agents feeding one collector with a memory channel on the collector. A collector restart loses every batch in flight, and the edge channels fill while it is down. Use a file channel on the collector and at least two collectors behind a failover sink group.

## Configurations

### A complete HDFS pipeline

Worked full pipeline: `taildir` source, file channel, HDFS sink, and a host interceptor, writing to time-partitioned directories:

```properties
a1.sources = r1
a1.channels = c1
a1.sinks = k1

a1.sources.r1.type = taildir
a1.sources.r1.positionFile = /var/lib/flume/weblogs_position.json
a1.sources.r1.filegroups = f1
a1.sources.r1.filegroups.f1 = /var/log/app/.*\\.log
a1.sources.r1.channels = c1
a1.sources.r1.interceptors = host
a1.sources.r1.interceptors.host.type = host

a1.channels.c1.type = file
a1.channels.c1.checkpointDir = /data/flume/checkpoint
a1.channels.c1.dataDirs = /data/flume/data
a1.channels.c1.capacity = 2000000
a1.channels.c1.transactionCapacity = 20000

a1.sinks.k1.type = hdfs
a1.sinks.k1.hdfs.path = hdfs://nn/logs/app/year=%Y/month=%m/day=%d/hour=%H
a1.sinks.k1.hdfs.fileType = DataStream
a1.sinks.k1.hdfs.rollSize = 268435456
a1.sinks.k1.hdfs.rollCount = 0
a1.sinks.k1.hdfs.rollInterval = 600
a1.sinks.k1.channel = c1
```

+ `rollCount = 0` disables count-based rolling so files are governed by size and time, which keeps file sizes predictable.
+ The `year=.../month=...` layout is Hive-friendly partition naming, so the downstream table can use `PARTITIONED BY (year, month, day, hour)`.
+ `rollInterval` must be shorter than the sink `callTimeout` tuning window; otherwise a stalled file accumulates events.
+ The file channel checkpoint directory and data directories should be on separate fast disks with enough free space for at least a few hours of events.

Common mistake: copying an HDFS sink block between streams without revisiting `rollSize` and `rollInterval`. A block tuned for 35 MB/s produces tiny files on a 1 MB/s stream, and NameNode small-file pressure is the usual root cause behind a vague report that "Flume is slow".

### Writing to Kafka

A Kafka sink wraps a `KafkaProducer`. `kafka.bootstrap.servers` points at the brokers, `kafka.topic` names the topic, and any `kafka.producer.*` property passes through to the producer. The sink commits to the channel only when the broker acknowledges according to the `acks` setting, so a broker outage backs events up in the channel instead of losing them.

Worked config and numbers. At 20,000 events/s with `batchSize = 2000` the sink sends 10 batches per second. With `acks=all` on three brokers every acknowledgement waits for the in-sync replicas, so `linger.ms` of 10 to 50 lets batches fill instead of paying a round trip per record.

```properties
a1.sinks.k1.type = org.apache.flume.sink.kafka.KafkaSink
a1.sinks.k1.kafka.bootstrap.servers = broker-1:9092,broker-2:9092,broker-3:9092
a1.sinks.k1.kafka.topic = logs.raw
a1.sinks.k1.kafka.producer.acks = all
a1.sinks.k1.kafka.producer.linger.ms = 20
a1.sinks.k1.kafka.producer.compression.type = lz4
a1.sinks.k1.batchSize = 2000
a1.sinks.k1.channel = c1
```

Common mistake: creating the topic with one partition. Flume can send 2,000 records per batch, but a single partition caps consumer parallelism at one, so downstream can never keep up. Size partitions for the consumers and choose the record key that defines ordering.

## Operating Flume

### Reliability and outage sizing

Reliability is three settings: a durable channel, a transactional sink, and a sink group when there is more than one destination. `capacity` sets the outage window, `transactionCapacity` sets the batch size, and `maxpenalty` plus `backoff` control how long a failed sink is skipped before it is retried.

Worked sizing. A 4 KB event at 10,000 events/s for a 30 minute outage needs 10000 x 1800 x 4 KB, which is about 72 GB per channel. A 5 GB channel covers only about two minutes at that rate. State the outage you promise and size to it rather than discovering the limit during an incident.

Common mistake: relying on a failover sink group to handle a poison record. Failover protects against a dead destination, not against an event that always fails to parse or write, which blocks the batch forever. Route a bounded number of retries to a dead-letter channel and then move on.

### Tuning capacity and heap

Tuning an agent is choosing where the bottleneck is allowed to live. A channel is a fixed-size buffer, so if the sink is slower than the source for longer than the buffer lasts, the source must slow down or the buffer overflows. Every tuning decision changes one of three things: the size of the buffer, the rate the sink can drain, or the size of the batch a transaction commits.

Three numbers work together. `capacity` is how many events can sit in the channel. `transactionCapacity` is the largest batch a source or sink may commit at once. Heap is the hard limit behind that: a memory channel must hold `capacity` events in the JVM, plus the in-flight source and sink batches. A common starting ratio is `transactionCapacity` at 1% to 5% of `capacity`, and a memory channel that needs 6 GB of payload needs far more than a default 1 GB heap.

Worked file-channel sizing. Event rate 30,000/s, average event 1.2 KB, outage to survive 20 minutes. The rate is 30000 x 1200 = 36,000,000 bytes/s, or 36 MB/s. Twenty minutes is 1200 seconds, so the channel must hold 36 MB/s x 1200 s = 43.2 GB. Divided by 1.2 KB per event that is 36,000,000 events, so set `capacity = 40000000` for headroom. A `transactionCapacity` of 100,000 keeps each commit at about 120 MB.

```properties
a1.channels.c1.type = file
a1.channels.c1.capacity = 40000000
a1.channels.c1.transactionCapacity = 100000
a1.channels.c1.checkpointInterval = 30000
a1.channels.c1.checkpointDir = /data/flume/checkpoint
a1.channels.c1.dataDirs = /data/flume/data
a1.channels.c1.fsyncPerTransaction = true
```

Reliability versus throughput is the same dial seen from two sides. `fsyncPerTransaction = true` makes every commit durable and caps throughput at the disk's fsync rate; `false` raises throughput and can lose the last few transactions on a power failure. A memory channel maximizes throughput and loses all buffered events on restart. Decide by event value: audit and billing keep `true`, rebuildable telemetry can use `false` or memory.

Common mistake: sizing `capacity` from memory instead of arithmetic. A channel set to 5,000,000 events looks generous until the average event is 4 KB, at which point it is 20 GB of payload and the agent dies on heap or disk. Compute events x average size first, then pick the channel type and the disk or heap that can actually hold it.

### Batching and sink throughput

Batch size is the main throughput lever because every transaction has a fixed cost: an RPC for Avro, a produce call for Kafka, a file append and an fsync for HDFS. Larger batches amortize that cost across more events. `batchSize` on the sink, `transactionCapacity` on the channel, and `batchSize` on the source are three names for the same batch on the three sides, and the smallest of them wins.

Worked throughput. An HDFS sink writing 1 KB events with an effective batch of 1,000 reaches 40 transactions/s, which is 40 MB/s. Raise the effective batch to 20,000 events and the same 40 MB/s needs only 2 transactions/s, a twentyfold cut in commit overhead. The bottleneck then moves to disk bandwidth and the HDFS client rather than to transaction cost.

```properties
a1.sources.r1.batchSize = 20000
a1.channels.c1.transactionCapacity = 20000

a1.sinks.k1.batchSize = 20000
a1.sinks.k1.hdfs.batchSize = 20000
a1.sinks.k1.hdfs.rollSize = 268435456
a1.sinks.k1.hdfs.rollInterval = 300
a1.sinks.k1.hdfs.rollCount = 0
```

The file channel is dominated by fsync and disk layout. Put `dataDirs` on high-IOPS storage, spread it across several directories on different devices so writes parallelize, and keep `checkpointDir` on a separate device from `dataDirs` so a checkpoint flush does not contend with data writes. `checkpointInterval` controls how much is replayed after an unclean shutdown, and `fsyncPerTransaction = false` trades the last transactions of a crash for throughput.

Common mistake: raising the sink `batchSize` without raising `transactionCapacity`. The sink asks for 20,000 events but the channel can only commit 5,000, so throughput stays flat while the sink logs channel exceptions or silently drains smaller batches. Set all three batch settings together and watch the average batch size in the metrics to confirm they agree.

### Monitoring

The HTTP reporting endpoint exposes counters per source, channel, and sink. The three to watch are `ChannelFillPercentage`, the gap between `EventPutSuccessCount` and `EventTakeSuccessCount`, and sink health through `EventDrainSuccessCount` and `BatchCompleteCount`.

```bash
# Start the agent with the HTTP reporting endpoint
flume-ng agent --conf /etc/flume/conf --conf-file /etc/flume/conf/weblogs.conf -n a1 \
  -Dflume.monitoring.type=http \
  -Dflume.monitoring.port=34545 \
  -Dflume.root.logger=INFO,console
```

Worked alert thresholds: fill above 70% for five minutes, a put-minus-take gap that widens for ten minutes, and a sink `EventDrainSuccessCount` that stops moving while puts continue. Add JVM heap and GC pause, because a memory channel lives entirely on heap and a large channel needs a large heap.

Common mistake: monitoring only the sink. A channel near 100% with a sink that looks alive often means the sink is failing transactions and rolling back, so puts succeed while takes never commit. Compare `EventTakeAttemptCount` with `EventTakeSuccessCount` to expose rollbacks.

### Diagnosing drops and duplicates

Drops and duplicates look different in the counters. A drop is a put that never becomes a successful take, or a source that never put at all. A duplicate is a take that reaches the destination twice because a transaction rolled back after the destination had already accepted the data. The metrics and the destination counts separate the two.

Start from the counters. `EventPutSuccessCount` minus `EventTakeSuccessCount` is the standing backlog; if the two converge again the events were delayed, not lost. `EventTakeAttemptCount` greater than `EventTakeSuccessCount` means rollbacks, which is the duplicate signal. A source whose put count stops moving has failed or is blocked by back-pressure from a full channel.

Worked duplicate diagnosis. The destination holds 1,080,000 rows for an hour in which the source wrote 1,000,000 events. The sink log shows 1,000 transactions and 40 retries, each retry re-sending a 2,000 event batch. The duplicates are 40 x 2000 = 80,000, exactly the gap. A timeout that rolled back after the write succeeded is the cause; an idempotency key or a shorter request timeout shrinks the ambiguous window.

Common mistake: blaming the sink for duplicates that are source re-reads. A `taildir` source without a `positionFile`, or a spooling directory whose files reappear, re-ingests the same lines on every restart. Check the source's position tracking and file counts before you touch the sink.

## Where Flume fits today

### Flume versus Kafka

Both are distributed, both move events reliably, and both can feed several destinations. The fundamental difference is push versus pull. Flume removes an event from its channel once a sink has delivered it; Kafka keeps the event on the broker until the retention window expires, so any consumer can re-read it.

Worked consequence. A Flume channel holds at most `capacity` events, and one slow sink blocks every consumer of that agent because there is one queue. A Kafka topic holds a retention window, and a slow consumer group lags without affecting the others. That is why Flume sits at the edge and Kafka sits in the middle.

Common mistake: describing Flume as "Kafka without retention". Flume also lacks broker-side replication of the channel, partition rebalancing, and cluster consensus. Those absences, not just the retention window, are the real tradeoffs to state in an interview.

### Flume versus Kafka Connect, Fluent Bit and Filebeat

| Tool | Model | Strength | Weakness |
|---|---|---|---|
| Flume | agent: source-channel-sink | durable HDFS/HBase sinks, mature Hadoop integration | JVM per agent, operational overhead, smaller ecosystem |
| Kafka Connect | workers plus connectors | declarative connectors, offsets managed by Kafka, huge connector library | needs Kafka as the backbone |
| Fluentd / Fluent Bit | plugin-driven collector | lightweight, cloud-native, strong filtering and parsers | HDFS/HBase sinks less common |
| Filebeat | shipper plus Logstash | tiny footprint on hosts, great for rotation-aware tailing | delivery and routing handled by a downstream tier |

The distinction that matters is where progress lives. Kafka Connect stores connector offsets in Kafka and is declarative. Flume stores progress in its own channel. Fluent Bit stores a position file and is built to be tiny. That choice decides how each tool recovers after a crash and who owns the offset.

Common mistake: choosing by popularity rather than by destination. When the sink is HDFS or HBase, Flume's sink and durable file channel are still the shortest path. When the sink is Kafka, a lighter connector avoids running and operating a JVM on every host.

### When not to use Flume

+ You need an event backbone with replay, many consumers, and long retention - use Kafka and consume with your own application or Kafka Connect.
+ You need per-event stream processing with state, windows, and joins - use Flink or Spark Structured Streaming.
+ You need simple, host-local log shipping with low resource use - use Fluent Bit or Filebeat.
+ You need database change capture from an operational system - use Debezium or a managed CDC service rather than a Flume SQL source.
+ You need exactly-once semantics into a lakehouse - land in Kafka and use a transactional sink (Iceberg or Delta).

Flume still fits a narrow, valuable role: durable, back-pressured collection of file and syslog-style events into HDFS or HBase on an existing Hadoop cluster.

Flume still pays off where the destination is HDFS or HBase and the environment is already Hadoop-centric. The valuable part is the durable local buffer next to each source, which a shared Kafka cluster does not give an edge host that loses its network for an hour. The channel turns a downstream outage into latency instead of data loss.

Common mistake: replacing Kafka with Flume on the assumption that a channel is a replayable log. A channel is a bounded per-agent queue with no consumer groups, no offsets, and no retention once a sink has drained it. If two teams need the same events later, land them in Kafka or object storage first.

Common mistake: keeping Flume because it already runs after the destination moved to a cloud warehouse. Flume's advantage is HDFS and HBase locality. Once the destination is object storage or a cloud warehouse, a managed ingestion service usually removes the agent fleet entirely.

## Real world use cases

### 1. Web and application log aggregation into HDFS

A media company runs 400 application servers producing 3 TB of access and application logs per day. Each host runs a lightweight Flume agent that tails the active log files and forwards over Avro to a tier of six collector agents, which write partitioned files to HDFS for Hive and Spark.

Business context: the analytics team needs daily and hourly log partitions without installing a heavy agent on every host. Data volume: 3 TB/day, roughly 35k events/second at peak.

Agent design: `taildir` source on the edge, file channel at the edge sized for 30 minutes of backlog, Avro sink to the collectors; collectors use a file channel and an HDFS sink with hour partitioning.

```properties
# Collector: partition by hour, roll at 128 MB or 10 minutes
col.sinks.k1.type = hdfs
col.sinks.k1.hdfs.path = hdfs://nn/logs/app/year=%Y/month=%m/day=%d/hour=%H
col.sinks.k1.hdfs.filePrefix = app-
col.sinks.k1.hdfs.fileType = DataStream
col.sinks.k1.hdfs.rollSize = 134217728
col.sinks.k1.hdfs.rollCount = 0
col.sinks.k1.hdfs.rollInterval = 600
col.sinks.k1.hdfs.round = true
col.sinks.k1.hdfs.roundValue = 10
col.sinks.k1.hdfs.roundUnit = minute
```

Why this design: two hops keep per-host memory small, the file channel at the edge absorbs collector restarts, and hour-level HDFS partitioning matches the Hive table's partition columns.

### 2. Multi-datacenter log fan-in with failover

A payments platform collects audit logs from three regions into a central HDFS archive. Each region has a primary and a secondary collector; edge agents must not lose events when a collector is down.

Business context: audit logs are regulatory evidence and must be durable. Data volume: 400 GB/day, bursty at settlement time.

Agent design: an Avro sink group with a `failover` processor, priorities set so the in-region collector is preferred, and a file channel with a multi-hour capacity on every edge agent.

```properties
edge.sinkgroups = g1
edge.sinkgroups.g1.sinks = k_primary k_secondary
edge.sinkgroups.g1.processor.type = failover
edge.sinkgroups.g1.processor.priority.k_primary = 10
edge.sinkgroups.g1.processor.priority.k_secondary = 5
edge.sinkgroups.g1.processor.maxpenalty = 120000
```

Why this design: failover preserves ordering per agent better than load balancing, the file channel survives an agent restart, and the two collectors can be in different racks or regions.

### 3. Security and audit log archival with zero loss

A bank must retain authentication and privilege-escalation logs for seven years. Agents run on domain controllers and application servers, and evidence of loss must be avoidable.

Business context: compliance requires an immutable, complete archive. Data volume: 80 GB/day but with a strict zero-loss requirement.

Agent design: a `taildir` source with `positionFile` for restart-safe tailing, a file channel with checkpointing, and an HDFS sink writing to a WORM-style directory with zstd or gzip compression and file names that include the collector host.

```properties
a1.sources.r1.type = taildir
a1.sources.r1.positionFile = /var/lib/flume/audit_position.json
a1.sources.r1.filegroups = audit
a1.sources.r1.filegroups.audit = /var/log/audit/.*\.log
```

Why this design: `taildir` with a position file never skips a line across restarts, the file channel turns a downstream outage into back-pressure instead of loss, and per-host file prefixes make it obvious when a host is missing from the archive.

## Flume exercises

The units match what Flume uses: events per second for rate, bytes for size, and events for channel capacity. Treat the asserted numbers as a check on the formula rather than a prediction of a real agent, because production adds serialization overhead, disk variance, and retries. The value is knowing which quantity to compute and which direction a change moves it.

### Try it 1. Size a memory channel from event rate and buffer time

**Task.** An agent receives 10,000 events/s, each about 1 KB, and must buffer 10 minutes without blocking the source. Compute the number of events, the raw payload size, and the heap to reserve if object overhead and in-flight batches roughly double the footprint.

**Hint.** `events = rate x seconds`, `bytes = events x 1024`, and `GiB = bytes / 1024**3`. Reserve twice the raw payload.

**Expected result.** `6,000,000` events, `5.72` GiB of raw payload, and about `11.44` GiB of heap.

In [ ]:
EVENT_BYTES = 1024
rate_per_s = 10_000
buffer_seconds = 600

events = rate_per_s * buffer_seconds
raw_bytes = events * EVENT_BYTES
raw_gib = raw_bytes / 1024 ** 3
heap_gib = raw_gib * 2

print(f"capacity      = {events:,} events")
print(f"raw payload   = {raw_bytes:,} bytes = {raw_gib:.2f} GiB")
print(f"heap to plan  = {heap_gib:.2f} GiB")

assert events == 6_000_000
assert round(raw_gib, 2) == 5.72
assert round(heap_gib, 2) == 11.44

### Try it 2. Choose a batch size from sink throughput

**Task.** The sink can sustain 40 MiB/s and each event is 2 KiB. To hold commits to 20 transactions per second, compute the events per second and the batch size, then round it up to a power of two.

**Hint.** `events_per_s = sink_bytes_per_s / event_bytes`; divide by the transaction target, then round up with `1 << (n - 1).bit_length()`.

**Expected result.** `20,480` events/s and a batch of `1,024` events, which is already a power of two.

In [ ]:
sink_bytes_per_s = 40 * 1024 * 1024
event_bytes = 2048
max_transactions_per_s = 20

events_per_s = sink_bytes_per_s / event_bytes
batch = events_per_s / max_transactions_per_s
batch_pow2 = 1 << (int(batch) - 1).bit_length()

print(f"sink rate        = {events_per_s:,.0f} events/s")
print(f"batch            = {batch:,.0f} events")
print(f"batch (power of 2) = {batch_pow2}")

assert events_per_s == 20480.0
assert batch == 1024.0
assert batch_pow2 == 1024

### Try it 3. Compare file and memory channels for durability

**Task.** An agent receives 20,000 events/s. Its memory channel can hold 5,000,000 events and its file channel can commit 20,000 events per transaction. Compute the seconds of data at risk for each, then pick the more durable option.

**Hint.** The memory channel risks everything it holds; the file channel risks only the in-flight batch, because committed transactions survive a restart.

**Expected result.** Memory risks `250.0` seconds; the file channel risks `1.0` second, so the file channel wins.

In [ ]:
rate_per_s = 20_000
memory_capacity = 5_000_000
file_transaction_capacity = 20_000

memory_loss_s = memory_capacity / rate_per_s
file_loss_s = file_transaction_capacity / rate_per_s
winner = "file channel" if file_loss_s < memory_loss_s else "memory channel"

print(f"memory channel risks {memory_loss_s:.1f} s of data")
print(f"file channel risks   {file_loss_s:.1f} s of data")
print("more durable:", winner)

assert memory_loss_s == 250.0
assert file_loss_s == 1.0
assert winner == "file channel"

### Try it 4. Compute duplicates under failure

**Task.** One million sink transactions each carry 5,000 events, and 0.1 percent of them roll back after the destination already accepted the write. Compute the rolled-back transactions, the duplicate events, and the duplicate ratio.

**Hint.** `rolled_back = transactions x rollback_rate`; every rolled-back transaction re-sends its whole batch.

**Expected result.** `1,000` rollbacks, `5,000,000` duplicate events, a ratio of `0.001`.

In [ ]:
transactions = 1_000_000
rollback_rate = 0.001
batch_events = 5_000

rolled_back = int(transactions * rollback_rate)
duplicates = rolled_back * batch_events
total_events = transactions * batch_events
duplicate_ratio = duplicates / total_events

print(f"rollbacks        = {rolled_back:,}")
print(f"duplicate events = {duplicates:,}")
print(f"duplicate ratio  = {duplicate_ratio:.4f}")

assert rolled_back == 1_000
assert duplicates == 5_000_000
assert round(duplicate_ratio, 4) == 0.001

### Try it 5. Size HDFS files from roll interval and size

**Task.** A stream writes 35 MiB/s. Compute the `rollSize` needed to produce 30 files per hour, then compute how many files per hour a 128 MiB `rollSize` would actually create.

**Hint.** `bytes_per_hour = rate x 3600`; `rollSize = bytes_per_hour / files_per_hour`.

**Expected result.** A roll size of `4200.0` MiB gives 30 files per hour, while a 128 MiB roll size gives about `984.38` files per hour, which is the small-file problem.

In [ ]:
rate_mib_per_s = 35
target_files_per_hour = 30
given_roll_size_mib = 128

bytes_per_hour_mib = rate_mib_per_s * 3600
roll_size_mib = bytes_per_hour_mib / target_files_per_hour
files_per_hour_given = bytes_per_hour_mib / given_roll_size_mib

print(f"bytes per hour      = {bytes_per_hour_mib:,.0f} MiB")
print(f"rollSize for 30/h   = {roll_size_mib:,.1f} MiB")
print(f"128 MiB rollSize    = {files_per_hour_given:,.2f} files/h")

assert roll_size_mib == 4200.0
assert round(files_per_hour_given, 2) == 984.38

### Try it 6. Diagnose a channel from fill percentage

**Task.** A channel with `capacity = 350,000` events recorded 1,500,000 successful puts and 1,200,000 successful takes since the agent started. Compute the standing backlog, the fill percentage, and decide whether the sink is the bottleneck.

**Hint.** `backlog = put - take` and `fill = backlog / capacity`. Treat fill above 70 percent as a sink bottleneck.

**Expected result.** A backlog of `300,000` events, `85.7` percent fill, and the verdict that the sink is the bottleneck.

In [ ]:
capacity = 350_000
put_success = 1_500_000
take_success = 1_200_000

backlog = put_success - take_success
fill_pct = backlog / capacity * 100
verdict = "sink is the bottleneck" if fill_pct > 70 else "healthy"

print(f"backlog = {backlog:,} events")
print(f"fill    = {fill_pct:.1f}%")
print("verdict:", verdict)

assert backlog == 300_000
assert round(fill_pct, 1) == 85.7
assert verdict == "sink is the bottleneck"

### Try it 7. Pick a transactionCapacity inside the healthy ratio

**Task.** A channel holds 2,000,000 events of 1 KiB and you want each commit to move about 50 MiB. Compute the batch in events, the ratio to `capacity`, and check whether it falls inside the recommended 1 to 5 percent band.

**Hint.** `batch_events = target_batch_mb x 1024**2 / event_bytes`; the ratio is `batch_events / capacity`.

**Expected result.** `51,200` events per batch, a ratio of `0.0256` (2.56 percent), which is inside the band.

In [ ]:
capacity = 2_000_000
event_bytes = 1024
target_batch_mib = 50

batch_events = int(target_batch_mib * 1024 ** 2 / event_bytes)
ratio = batch_events / capacity
within_band = 0.01 <= ratio <= 0.05

print(f"transactionCapacity = {batch_events:,} events")
print(f"ratio to capacity   = {ratio:.4f}")
print("within 1-5%:", within_band)

assert batch_events == 51_200
assert round(ratio, 4) == 0.0256
assert within_band is True

### Try it 8. Size a spillable channel for a burst

**Task.** A spillable memory channel buffers 500,000 events on heap and can spill another 2,000,000 to disk. At a burst of 4,000 events/s, compute how long until spilling starts and how long until the disk overflow is exhausted.

**Hint.** Divide each capacity by the burst rate; the total is the heap capacity plus the overflow capacity.

**Expected result.** Spilling starts after `125.0` seconds and the overflow is exhausted after `625.0` seconds.

In [ ]:
heap_capacity = 500_000
overflow_capacity = 2_000_000
burst_rate = 4_000

spill_after_s = heap_capacity / burst_rate
disk_after_s = (heap_capacity + overflow_capacity) / burst_rate

print(f"spill starts after      {spill_after_s:.1f} s")
print(f"disk overflow exhausted {disk_after_s:.1f} s")

assert spill_after_s == 125.0
assert disk_after_s == 625.0